In [23]:
import os
import csv
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader


CSV_PATH = "dataset_split.csv"  
SPLIT_TRAIN = "train"
SPLIT_VAL   = "val"
SPLIT_TEST  = "test"

ORIGINAL_SHAPE = (256, 256)    
DOWNSAMPLE_TO  = (64, 64)        # Downsample frames to this size (width, height)

FRAMES_PER_VIDEO = 120           

# Model / Training settings
INPUT_DIM   = DOWNSAMPLE_TO[0] * DOWNSAMPLE_TO[1] * 3   # e.g., 64*64*3 = 12288
HIDDEN_DIM  = 256
NUM_EPOCHS  = 20
BATCH_SIZE  = 4
LEARNING_RATE = 5e-4


In [24]:

class RawVideoDataset(Dataset):

    def __init__(self, csv_file, split="train", resize_shape=(64, 64)):
        """
        Args:
            csv_file (str): Path to the CSV that has columns: 'filepath', 'label', 'split'.
            split (str): Which split to load ('train', 'val', or 'test').
            resize_shape (tuple): (width, height) to resize each frame.
        """
        self.samples = []
        self.resize_shape = resize_shape
        
        with open(csv_file, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] == split:
                    self.samples.append((row["filepath"], row["label"]))
        
        unique_labels = sorted(list(set([s[1] for s in self.samples])))
        self.label_to_idx = {lbl: i for i, lbl in enumerate(unique_labels)}
        

        self.samples = [(fp, self.label_to_idx[lab]) for (fp, lab) in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Returns:
            frames_tensor: shape (120, input_dim), where input_dim = width*height*3
            label_idx: int, the label of this sequence
        """
        npy_path, label_idx = self.samples[idx]
        
        frames = np.load(npy_path) 
        
        # Downsample + flatten each frame
        processed_frames = []
        for frame in frames:
            resized_frame = cv2.resize(frame, self.resize_shape)  # shape: (64, 64, 3)
            
            
            # Flatten to 1D
            flat_frame = resized_frame.reshape(-1)  # shape: (64*64*3,)
            processed_frames.append(flat_frame)
        
        # Stack into shape (120, input_dim)
        processed_frames = np.array(processed_frames, dtype=np.float32)
        
        # Convert to torch tensor
        frames_tensor = torch.from_numpy(processed_frames)  # shape = (120, input_dim)
        
        return frames_tensor, label_idx



In [25]:


class VideoLSTM(nn.Module):
    """
    A simple LSTM model that takes (batch, seq_len=120, input_dim)
    and outputs a classification over the gesture label.
    """
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(VideoLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=hidden_dim,
                            num_layers=2,  # Stacked LSTMs
                            dropout=0.3,  # Dropout for regularization
                            bidirectional=True,  # Use bidirectional LSTM
                            batch_first=True)
        self.fc   = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x):
        """
        x.shape = (batch, 120, input_dim)
        """
        out, (h_n, c_n) = self.lstm(x)  
        
        last_out = out[:, -1, :]      
        
        logits = self.fc(last_out)     
        return logits


In [26]:

def train_direct_lstm():
    train_dataset = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_TRAIN, resize_shape=DOWNSAMPLE_TO)
    val_dataset   = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_VAL,   resize_shape=DOWNSAMPLE_TO)
    test_dataset  = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_TEST,  resize_shape=DOWNSAMPLE_TO)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
    
    num_classes = len(train_dataset.label_to_idx)
    
    model = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=num_classes)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    for epoch in range(NUM_EPOCHS):

        model.train()
        total_loss = 0.0
        for frames_batch, labels_batch in train_loader:

            frames_batch = frames_batch.to(device)
            labels_batch = labels_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(frames_batch) 
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_loader)
        

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for frames_batch, labels_batch in val_loader:
                frames_batch = frames_batch.to(device)
                labels_batch = labels_batch.to(device)
                
                outputs = model(frames_batch)
                loss = criterion(outputs, labels_batch)
                val_loss += loss.item()
                
                _, preds = torch.max(outputs, dim=1)
                correct += (preds == labels_batch).sum().item()
                total += labels_batch.size(0)
        
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_accuracy = (correct / total) if total > 0 else 0
        
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {val_accuracy*100:.2f}%")
    
    model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for frames_batch, labels_batch in test_loader:
            frames_batch = frames_batch.to(device)
            labels_batch = labels_batch.to(device)
            outputs = model(frames_batch)
            loss = criterion(outputs, labels_batch)
            test_loss += loss.item()
            
            _, preds = torch.max(outputs, dim=1)
            test_correct += (preds == labels_batch).sum().item()
            test_total += labels_batch.size(0)
    avg_test_loss = test_loss / len(test_loader) if len(test_loader) > 0 else 0
    test_accuracy = (test_correct / test_total) if test_total > 0 else 0
    print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_accuracy*100:.2f}%")

    torch.save(model.state_dict(), "lstm_model.pth")
    print("Model saved.")

if __name__ == "__main__":
    train_direct_lstm()



Epoch [1/20] | Train Loss: 2.9426 | Val Loss: 2.8036 | Val Acc: 15.00%
Epoch [2/20] | Train Loss: 2.7332 | Val Loss: 2.7787 | Val Acc: 13.33%
Epoch [3/20] | Train Loss: 2.6649 | Val Loss: 2.7415 | Val Acc: 13.33%
Epoch [4/20] | Train Loss: 2.6351 | Val Loss: 2.7327 | Val Acc: 16.67%
Epoch [5/20] | Train Loss: 2.6161 | Val Loss: 2.7207 | Val Acc: 18.33%
Epoch [6/20] | Train Loss: 2.5942 | Val Loss: 2.7522 | Val Acc: 18.33%
Epoch [7/20] | Train Loss: 2.5669 | Val Loss: 2.7204 | Val Acc: 20.00%
Epoch [8/20] | Train Loss: 2.5593 | Val Loss: 2.7328 | Val Acc: 16.67%
Epoch [9/20] | Train Loss: 2.5374 | Val Loss: 2.7250 | Val Acc: 18.33%
Epoch [10/20] | Train Loss: 2.5352 | Val Loss: 2.7103 | Val Acc: 18.33%
Epoch [11/20] | Train Loss: 2.5023 | Val Loss: 2.7313 | Val Acc: 20.00%
Epoch [12/20] | Train Loss: 2.5070 | Val Loss: 2.7889 | Val Acc: 16.67%
Epoch [13/20] | Train Loss: 2.4987 | Val Loss: 2.7675 | Val Acc: 20.00%
Epoch [14/20] | Train Loss: 2.4679 | Val Loss: 2.7913 | Val Acc: 20.00%
E

In [27]:
model = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=20)
model.load_state_dict(torch.load("lstm_model.pth"))
model.eval()


/var/folders/js/b36xj6rs3k1c5z78dsb4p3p00000gn/T/ipykernel_83118/4115478751.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("lstm_model.

VideoLSTM(
  (lstm): LSTM(12288, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=512, out_features=20, bias=True)
)

In [28]:
import cv2
import torch
import numpy as np

def predict_video(video_npy_path, model, label_map, resize_shape=(64,64), device='cpu'):
    """
    video_npy_path: path to the .npy file of shape (120, H, W, 3)
    model: your trained LSTM model (already in eval mode)
    label_map: a dict mapping label_idx -> label_name, e.g. {0: "hello", 1: "thanks", ...}
    resize_shape: same as used in your RawVideoDataset
    device: 'cpu' or 'cuda'
    """
    # 1. Load frames
    frames = np.load(video_npy_path)  # shape (120, H, W, 3)
    
    # 2. Downsample + flatten each frame just like in your RawVideoDataset
    processed_frames = []
    for frame in frames:
        # Downsample
        small_frame = cv2.resize(frame, resize_shape)  # shape = (64,64,3)
        # Flatten
        flat_frame = small_frame.reshape(-1)  # shape = (64*64*3,)
        processed_frames.append(flat_frame)
    
    # 3. Convert to numpy array -> torch tensor
    processed_frames = np.array(processed_frames, dtype=np.float32)  # shape (120, 64*64*3)
    processed_tensor = torch.from_numpy(processed_frames).unsqueeze(0)  # shape (1, 120, input_dim)
    
    # 4. Move to device
    processed_tensor = processed_tensor.to(device)
    
    # 5. Model forward
    with torch.no_grad():
        outputs = model(processed_tensor)  # shape (1, num_classes)
    
    # 6. Prediction
    _, pred_idx = torch.max(outputs, dim=1)  # shape (1,)
    pred_idx = pred_idx.item()  # get the integer
    pred_label = label_map[pred_idx]
    
    return pred_label


In [37]:
import csv
model = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=20)
model.load_state_dict(torch.load("lstm_model.pth"))
model.eval()
# Function to recreate label_to_idx
def load_label_to_idx(csv_file):
    """
    Reads the CSV and creates the label_to_idx mapping.
    """
    with open(csv_file, "r") as f:
        reader = csv.DictReader(f)
        labels = set(row["label"] for row in reader)
    label_to_idx = {lbl: idx for idx, lbl in enumerate(sorted(labels))}
    return label_to_idx

# Load the mapping
label_to_idx = load_label_to_idx(CSV_PATH)

# Reverse the mapping to get idx_to_label
idx_to_label = {idx: lbl for lbl, idx in label_to_idx.items()}

# Test your video
#sample_video_path = "processed_frames_test/class/class-test.npy"  # Replace with your test file path
#sample_video_path = "processed_frames_test/room/room-test.npy"  # for example
sample_video_path = "processed_frames_test/teach/teach-test.npy"  # for example
# Ensure the model is in evaluation mode
model.eval()

predicted_label = predict_video(
    video_npy_path=sample_video_path,
    model=model,
    label_map=idx_to_label,
    resize_shape=DOWNSAMPLE_TO,
    device='cpu'  # Change to 'cuda' if using GPU
)

print("Predicted label:", predicted_label)



Predicted label: train


/var/folders/js/b36xj6rs3k1c5z78dsb4p3p00000gn/T/ipykernel_83118/923908753.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("lstm_model.p